In [ ]:
# ============================================================================
# 01_train_and_evaluate.ipynb
# ============================================================================
# PURPOSE
#   Run experiments and save results to Drive.
#   Each section is independent — run only what you need.
#
# RULES
#   - Run Section 0 at the start of every session (30 seconds)
#   - Run a section only when you want to run that experiment
#   - Results save to Drive automatically after each section
#   - Never need to re-run a completed section
#
# SECTIONS
#   0  Setup and helpers         — run every session
#   1  Train and serialize       — ONE TIME: produces joblib files
#   2  Hybrid pipeline           — ONE TIME: ~60 min on GPU
#   3  Qwen2.5-1.5B evaluation   — ONE TIME: ~8 min on GPU
#   4  Qwen2.5-3B evaluation     — ONE TIME: ~14 min on GPU
#   5  Qwen2.5-7B evaluation     — ONE TIME: ~25 min on GPU
#   6  Claude Haiku evaluation   — ONE TIME: costs API credits
#   7  Groq evaluation           — ONE TIME: 300-report subset
#   8  Add new model             — template for future models
# ============================================================================

In [ ]:
# ── Section 0: Setup and helpers ─────────────────────────────────────────────
# Run every session before anything else. Takes ~30 seconds.

import gc, os, pickle, random, time
import numpy as np
import pandas as pd
import joblib
import torch

from collections import Counter
from pathlib import Path
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.svm import LinearSVC
from transformers import pipeline as hf_pipeline, AutoTokenizer
from google.colab import drive

drive.mount("/content/drive")

# Paths
PROJECT_DIR = Path("/content/drive/MyDrive/LLM_to_API_Project")
DATA_DIR    = PROJECT_DIR / "data"
MODELS_DIR  = PROJECT_DIR / "models"
EVAL_DIR    = PROJECT_DIR / "eval_results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = EVAL_DIR / "hybrid_checkpoint.pkl"

# Config — locked, never change
SEED                 = 42
TOP_K                = 5
CHUNK_TOKENS         = 256
CHUNK_STRIDE         = 64
MAX_CHUNKS           = 12
CONFIDENCE_THRESHOLD = 0.6

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

LABEL_MAP = {
    "ACC":"adrenocortical carcinoma","BLCA":"bladder urothelial carcinoma",
    "BRCA":"breast invasive carcinoma",
    "CESC":"cervical squamous cell carcinoma and endocervical adenocarcinoma",
    "CHOL":"cholangiocarcinoma","COAD":"colon adenocarcinoma",
    "DLBC":"diffuse large B-cell lymphoma","ESCA":"esophageal carcinoma",
    "GBM":"glioblastoma multiforme","HNSC":"head and neck squamous cell carcinoma",
    "KICH":"kidney chromophobe","KIRC":"kidney renal clear cell carcinoma",
    "KIRP":"kidney renal papillary cell carcinoma","LAML":"acute myeloid leukemia",
    "LGG":"brain lower grade glioma","LIHC":"liver hepatocellular carcinoma",
    "LUAD":"lung adenocarcinoma","LUSC":"lung squamous cell carcinoma",
    "MESO":"mesothelioma","OV":"ovarian serous cystadenocarcinoma",
    "PAAD":"pancreatic adenocarcinoma","PCPG":"pheochromocytoma and paraganglioma",
    "PRAD":"prostate adenocarcinoma","READ":"rectum adenocarcinoma","SARC":"sarcoma",
    "SKCM":"skin cutaneous melanoma","STAD":"stomach adenocarcinoma",
    "TGCT":"testicular germ cell tumors","THCA":"thyroid carcinoma","THYM":"thymoma",
    "UCEC":"uterine corpus endometrial carcinoma","UCS":"uterine carcinosarcoma",
    "UVM":"uveal melanoma",
}


def focus_text(report, max_chars=2000):
    if report is None: return ""
    txt = str(report)
    for anchor in ["final diagnosis","pathologic diagnosis","diagnosis",
                   "impression","comment","microscopic description"]:
        pos = txt.lower().find(anchor)
        if pos != -1: return txt[pos:pos+max_chars]
    return txt[:max_chars]


def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return {}
    with open(CHECKPOINT_PATH, "rb") as f:
        return pickle.load(f)


def save_checkpoint(ck):
    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(ck, f)
    print(f"Saved to checkpoint. Keys: {list(ck.keys())}")


def run_inference(texts, topk_list, call_fn, label):
    preds = []
    t0    = time.time()
    for i, (text, top_k) in enumerate(zip(texts, topk_list)):
        preds.append(call_fn(text, top_k, i))
        if (i + 1) % 50 == 0:
            elapsed = time.time() - t0
            eta     = (len(texts) - i - 1) / ((i + 1) / elapsed)
            print(f"  [{i+1}/{len(texts)}]  "
                  f"elapsed: {elapsed/60:.1f}min  ETA: {eta/60:.1f}min")
    total = time.time() - t0
    print(f"{label} done: {total/60:.1f} min  ({total/len(texts):.2f} sec/report)")
    return preds


def free_gpu():
    global hf_model, hf_tokenizer
    try:
        del hf_model, hf_tokenizer
    except NameError:
        pass
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")


def call_hf_model(text, candidates, report_idx):
    focused    = focus_text(text)
    label_list = "\n".join([f"- {c}: {LABEL_MAP[c]}" for c in candidates])
    prompt = (
        "You are a clinical pathology expert.\n\n"
        "Read the following pathology report excerpt and classify it "
        "into exactly one cancer type from the list below.\n\n"
        f"REPORT:\n{focused}\n\n"
        f"CANCER TYPE OPTIONS:\n{label_list}\n\n"
        "Respond with ONLY the cancer type code. No explanation. Just the code."
    )
    inputs = hf_tokenizer(
        hf_tokenizer.apply_chat_template(
            [{"role":"user","content":prompt}],
            tokenize=False, add_generation_prompt=True
        ),
        return_tensors="pt"
    ).to(hf_model.device)
    try:
        with torch.no_grad():
            out = hf_model.generate(
                **inputs, max_new_tokens=10, do_sample=False,
                pad_token_id=hf_tokenizer.eos_token_id
            )
        raw  = hf_tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip().upper()
        code = raw.split()[0] if raw else candidates[0]
        return code if code in candidates else candidates[0]
    except Exception as e:
        print(f"  Error on {report_idx}: {e}")
        return candidates[0]


print("Setup complete ✅")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")
if CHECKPOINT_PATH.exists():
    ck = load_checkpoint()
    print(f"Keys in checkpoint: {list(ck.keys())}")

In [ ]:
# ── Section 1: Train and serialize models ────────────────────────────────────
# ONE TIME — produces vectorizer.joblib, svm_model.joblib, lr_model.joblib
# Skip this section if models/ folder already has the joblib files.

reports_df = pd.read_csv(DATA_DIR / "TCGA_Reports.csv")
reports_df["patient_id"] = reports_df["patient_filename"].astype(str).str.split(".").str[0]
meta_df    = pd.read_csv(DATA_DIR / "tcga_patient_to_cancer_type.csv")

tcga_df = (
    reports_df.merge(meta_df, on="patient_id", how="inner")
    .dropna(subset=["text","cancer_type"]).reset_index(drop=True)
)
print(f"Loaded {len(tcga_df):,} reports | {tcga_df['cancer_type'].nunique()} types")

X, y, groups = tcga_df["text"], tcga_df["cancer_type"], tcga_df["patient_id"]
gss          = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])) == 0
print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Leakage: none ✓")

NEGATION_KEEP = {"no","not","nor","never","without"}
stopwords     = sorted(set(ENGLISH_STOP_WORDS) - NEGATION_KEEP)

vectorizer    = TfidfVectorizer(stop_words=stopwords, max_features=50_000,
                                ngram_range=(1,2), min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print("Training Logistic Regression...")
lr_model = LogisticRegression(solver="saga", max_iter=6000,
                               class_weight="balanced", random_state=SEED, C=4.0)
lr_model.fit(X_train_tfidf, y_train)

print("Training LinearSVC (calibrated)...")
svm_model = CalibratedClassifierCV(
    LinearSVC(random_state=SEED, class_weight="balanced"), cv=5
)
svm_model.fit(X_train_tfidf, y_train)

for name, model in [("LogReg", lr_model), ("LinearSVC", svm_model)]:
    preds = model.predict(X_test_tfidf)
    print(f"{name}: acc={accuracy_score(y_test,preds):.4f}  "
          f"macro-F1={f1_score(y_test,preds,average='macro'):.4f}")

joblib.dump(vectorizer, MODELS_DIR / "vectorizer.joblib")
joblib.dump(lr_model,   MODELS_DIR / "lr_model.joblib")
joblib.dump(svm_model,  MODELS_DIR / "svm_model.joblib")
print(f"Models saved to {MODELS_DIR}/ ✅")

In [ ]:
# ── Section 2: Hybrid pipeline on full test set ──────────────────────────────
# ONE TIME — takes ~60 minutes on GPU.
# Skip if hybrid_checkpoint.pkl already exists on Drive.

vectorizer = joblib.load(MODELS_DIR / "vectorizer.joblib")
svm_model  = joblib.load(MODELS_DIR / "svm_model.joblib")

X_eval = X_test.reset_index(drop=True)
y_eval = y_test.reset_index(drop=True)

label_set       = sorted(y_eval.unique())
verbose_to_code = {LABEL_MAP[c]: c for c in label_set}


def get_topk(tfidf_row, k=TOP_K):
    base    = svm_model.calibrated_classifiers_[0].estimator
    scores  = np.asarray(base.decision_function(tfidf_row)).reshape(-1)
    top_idx = np.argsort(scores)[-k:][::-1]
    return [svm_model.classes_[i] for i in top_idx]


def chunk_by_tokens(text, tokenizer):
    ids, chunks, start = (
        tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"],
        [], 0
    )
    while start < len(ids) and len(chunks) < MAX_CHUNKS:
        end = min(start + CHUNK_TOKENS, len(ids))
        chunks.append(tokenizer.decode(ids[start:end], skip_special_tokens=True))
        if end == len(ids): break
        start = max(0, end - CHUNK_STRIDE)
    return chunks if chunks else [str(text)]


device   = 0 if torch.cuda.is_available() else -1
nli_pipe = hf_pipeline("zero-shot-classification",
                        model="facebook/bart-large-mnli", device=device)
nli_tok  = AutoTokenizer.from_pretrained("facebook/bart-large-mnli", use_fast=True)
print(f"NLI model loaded on {'GPU' if device==0 else 'CPU'}")

hybrid_preds, hybrid_confidences, hybrid_topk_list = [], [], []
t0 = time.time()

for i, text in enumerate(X_eval.tolist()):
    tfidf_vec    = vectorizer.transform([text])
    top_k        = get_topk(tfidf_vec)
    cand_verbose = [LABEL_MAP[c] for c in top_k]
    best_code, best_score = top_k[0], -1.0

    for chunk in chunk_by_tokens(text, nli_tok):
        out = nli_pipe(chunk, candidate_labels=cand_verbose,
                       hypothesis_template="This pathology report describes {}.",
                       truncation=True, max_length=1024)
        if float(out["scores"][0]) > best_score:
            best_score = float(out["scores"][0])
            best_code  = verbose_to_code[out["labels"][0]]

    hybrid_preds.append(best_code)
    hybrid_confidences.append(best_score)
    hybrid_topk_list.append(top_k)

    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f"  [{i+1}/{len(X_eval)}]  "
              f"elapsed: {elapsed/60:.1f}min  "
              f"ETA: {(len(X_eval)-i-1)/((i+1)/elapsed)/60:.1f}min")

print(f"Hybrid done: {(time.time()-t0)/60:.1f} min")
print(f"Accuracy: {accuracy_score(y_eval,hybrid_preds):.4f}  "
      f"macro-F1: {f1_score(y_eval,hybrid_preds,average='macro',zero_division=0):.4f}")

conf_arr = np.array(hybrid_confidences)
print(f"High confidence (≥{CONFIDENCE_THRESHOLD}): "
      f"{(conf_arr>=CONFIDENCE_THRESHOLD).mean()*100:.1f}%")
print(f"Low confidence  (<{CONFIDENCE_THRESHOLD}): "
      f"{(conf_arr<CONFIDENCE_THRESHOLD).mean()*100:.1f}%")

ck = load_checkpoint()
ck.update({
    "X_eval_texts"       : X_eval.tolist(),
    "y_eval_labels"      : y_eval.tolist(),
    "hybrid_preds"       : hybrid_preds,
    "hybrid_confidences" : hybrid_confidences,
    "hybrid_topk_list"   : hybrid_topk_list,
})
save_checkpoint(ck)

In [ ]:
# ── Section 3: Qwen2.5-1.5B evaluation ──────────────────────────────────────
# ONE TIME — ~8 min on GPU.
# Skip if "qwen15b_preds" is already in checkpoint (check Section 0 output).

from transformers import AutoTokenizer, AutoModelForCausalLM

ck = load_checkpoint()

free_gpu()
hf_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
hf_model     = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16, device_map="auto"
)
hf_model.eval()

qwen15b_preds       = run_inference(ck["X_eval_texts"], ck["hybrid_topk_list"],
                                    call_hf_model, "Qwen2.5-1.5B")
ck["qwen15b_preds"] = qwen15b_preds
save_checkpoint(ck)

In [ ]:
# ── Section 4: Qwen2.5-3B evaluation ────────────────────────────────────────
# ONE TIME — ~14 min on GPU.
# Skip if "qwen3b_preds" is already in checkpoint.

from transformers import AutoTokenizer, AutoModelForCausalLM

ck = load_checkpoint()

free_gpu()
hf_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
hf_model     = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", torch_dtype=torch.float16, device_map="auto"
)
hf_model.eval()

qwen3b_preds       = run_inference(ck["X_eval_texts"], ck["hybrid_topk_list"],
                                   call_hf_model, "Qwen2.5-3B")
ck["qwen3b_preds"] = qwen3b_preds
save_checkpoint(ck)

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1"

In [ ]:

# ── Section 5: Qwen2.5-7B evaluation (4-bit quantized) ──────────────────────
# ONE TIME — ~25 min on GPU.
# Skip if "qwen7b_preds" is already in checkpoint.
# Requires bitsandbytes — run the pip install cell below first.



from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

ck = load_checkpoint()

free_gpu()
hf_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
hf_model     = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4"
    ),
    device_map="auto"
)
hf_model.eval()

qwen7b_preds       = run_inference(ck["X_eval_texts"], ck["hybrid_topk_list"],
                                   call_hf_model, "Qwen2.5-7B")
ck["qwen7b_preds"] = qwen7b_preds
save_checkpoint(ck)

In [ ]:
!pip install anthropic

In [ ]:
# ── Section 6: Claude Haiku evaluation ──────────────────────────────────────
# ONE TIME — costs API credits (~$0.50 for 1905 reports).
# Skip if "haiku_preds" is already in checkpoint.
# Set ANTHROPIC_API_KEY in your .env or paste below.

# !pip install -q anthropic

import anthropic
from google.colab import userdata

ck               = load_checkpoint()
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
client           = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def call_haiku(text, candidates, report_idx):
    focused    = focus_text(text)
    label_list = "\n".join([f"- {c}: {LABEL_MAP[c]}" for c in candidates])
    prompt = (
        "You are a clinical pathology expert.\n\n"
        "Classify the following pathology report into exactly one cancer "
        "type from the list below.\n\n"
        f"REPORT:\n{focused}\n\n"
        f"OPTIONS:\n{label_list}\n\n"
        "Respond with ONLY the cancer type code. Just the code."
    )
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001", max_tokens=10,
            messages=[{"role":"user","content":prompt}]
        )
        raw  = response.content[0].text.strip().upper()
        code = raw.split()[0] if raw else candidates[0]
        return code if code in candidates else candidates[0]
    except Exception as e:
        print(f"  Error on {report_idx}: {e}")
        return candidates[0]


haiku_preds       = run_inference(ck["X_eval_texts"], ck["hybrid_topk_list"],
                                  call_haiku, "Claude Haiku")
ck["haiku_preds"] = haiku_preds
save_checkpoint(ck)

In [ ]:
# ── Section 7: Groq Llama 3.1 8B evaluation (N=300 subset) ──────────────────
# ONE TIME — free, but limited to 500K tokens/day on Groq free tier.
# Uses N=300 fixed subset. Skip if "groq_sub_preds" is already in checkpoint.
# Note: Full 1905-report evaluation hit the daily token limit at ~886 reports.
!pip install -q groq

from groq import Groq

ck              = load_checkpoint()
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
groq_client  = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL      = "llama-3.1-8b-instant"

rng     = np.random.RandomState(SEED)
sub_idx = rng.choice(len(ck["X_eval_texts"]), size=300, replace=False)


def call_groq(text, candidates, report_idx):
    focused    = focus_text(text)
    label_list = "\n".join([f"- {c}: {LABEL_MAP[c]}" for c in candidates])
    prompt = (
        "You are a clinical pathology expert.\n\n"
        "Classify the following pathology report into exactly one cancer "
        "type from the list below.\n\n"
        f"REPORT:\n{focused}\n\n"
        f"OPTIONS:\n{label_list}\n\n"
        "Respond with ONLY the cancer type code. Just the code."
    )
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role":"user","content":prompt}],
            max_tokens=10, temperature=0
        )
        raw  = response.choices[0].message.content.strip().upper()
        code = raw.split()[0] if raw else candidates[0]
        return code if code in candidates else candidates[0]
    except Exception as e:
        print(f"  Groq error on {report_idx}: {e}")
        return candidates[0]


X_sub    = [ck["X_eval_texts"][i]    for i in sub_idx]
topk_sub = [ck["hybrid_topk_list"][i] for i in sub_idx]

groq_sub_preds       = run_inference(X_sub, topk_sub, call_groq, "Groq Llama3.1-8B")
ck["groq_sub_preds"] = groq_sub_preds
ck["groq_sub_idx"]   = sub_idx.tolist()
save_checkpoint(ck)

In [ ]:
# ── Section 8: Template for new models ───────────────────────────────────────
# Copy this section to evaluate any new model.
# Change: MODEL_NAME, checkpoint_key, and the section header.
# Everything else stays the same.

# from transformers import AutoTokenizer, AutoModelForCausalLM
#
# ck           = load_checkpoint()
# MODEL_NAME   = "YOUR_MODEL_HERE"
# CHECKPOINT_KEY = "your_model_preds"   # unique key for this model
#
# free_gpu()
# hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# hf_model     = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
# )
# hf_model.eval()
#
# new_preds            = run_inference(ck["X_eval_texts"], ck["hybrid_topk_list"],
#                                      call_hf_model, MODEL_NAME)
# ck[CHECKPOINT_KEY]   = new_preds
# save_checkpoint(ck)